## GenAI Tracing with MLflow

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    openai-agents==0.22.0 \
    mcp==2.0.0 \
    databricks-mcp==0.9.2 \
    "mlflow>=3.1"

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Set up the Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Set up MLflow Tracing

In [ ]:
import mlflow
import os

# Enable auto-tracing for OpenAI
mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/Trace-LLM-Application")

### Define your MCP Server URL

In [ ]:
from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient


custom_mcp_server_url = "YOUR-CUSTOM-MCP-SERVER-URL-GOES-HERE"

code_interpreter_mcp_server_url = (
    f"{workspace_host}/api/2.0/mcp/functions/"
    f"system/ai/python_exec"
)

### Define the Agent

In [ ]:
from agents import (
    Agent,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)
from agents.mcp import MCPServerStreamableHttp

@mlflow.trace
async def run_agent(user_query: str):
    # Create an OpenAI-compatible client for Databricks
    client = AsyncOpenAI(
        api_key=token,
        base_url=f"{workspace_host}/serving-endpoints"
    )

    # Configure the Databricks model
    model = OpenAIChatCompletionsModel(
        model="databricks-claude-sonnet-4-5",
        openai_client=client
    )

    async with (
        MCPServerStreamableHttp(
            name="MSLearn-MCP-Server",
            params={
                "url": custom_mcp_server_url,
                "headers": {
                    "Authorization": f"Bearer {token}"
                }
            }
        ) as custom_mcp_server
    ):

        agent = Agent(
            name="MS-Learn-Agent",

            instructions="""
            You are a helpful AI assistant with access to the MS Learn MCP server

            Use Microsoft Learn MCP for:
            - Microsoft technical documentation
            - Azure documentation
            - Azure Databricks documentation
            - Microsoft learning resources
            """,

            model=model,

            mcp_servers=[
                custom_mcp_server
            ]
        )

        result = await Runner.run(
            agent,
            user_query
        )

        return result.final_output
    

### Define the Prediction Function

In [ ]:
import nest_asyncio
import asyncio

nest_asyncio.apply()

def predict_fn(query: str) -> str:
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(run_agent(query))

### Execute and Trace your Agent

In [ ]:
print(predict_fn("Help me with some learning paths for the Microsoft AI-103: Azure AI Apps and Agents Developer Associate Certification"))